In [8]:
# Client Initialization and helper functions

# Imports
import os
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
import anthropic
from datetime import datetime

# Unset proxy env vars before creating the client
for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy", "ALL_PROXY", "all_proxy"]:
    os.environ.pop(var, None)

# Explicitly bypass proxy for localhost
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"

env_path = ".env"
load_dotenv(dotenv_path=env_path, override=True)

client = anthropic.Anthropic(
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

llm_model = "llama3.2"

In [5]:
# helper functions
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "user",
        "content": text
    }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": llm_model,
        "max_tokens": 1024,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [14]:
from anthropic.types import ToolParam

# tool function to gett current datetime
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Get the current local date and time formatted as a string using a Python strftime-compatible format. Use this when the user asks for the current date, current time, current datetime, or asks to format the current datetime in a specific way. Do not use this tool if the user only wants a past or future date calculation unrelated to the current datetime.",
  "strict": True,
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "Optional Python strftime format string used to format the current datetime, for example '%Y-%m-%d %H:%M:%S', '%Y-%m-%d', or '%I:%M %p'. Must be a non-empty string when provided.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": [],
    "additionalProperties": False
  }
})

In [18]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formatted as HH:MM:SS?"
    }
)

response = client.messages.create(
    model=llm_model,
    max_tokens=1024,
    messages=messages,
    tools=[get_current_datetime_schema]
)

messages.append(
    {
        "role": "assistant",
        "content": response.content
    }
)

print(messages)

# print(response.content)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='call_7u3zruam', caller=None, input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]
